<h3>Adding the adapters to the base LLM for inference</h3>

In [ ]:
from unsloth import FastLanguageModel
from transformers import AutoTokenizer
from peft import PeftModel
import os
base_model_name="unsloth/Qwen3-4B-unsloth-bnb-4bit"
output_dir='./results'

base_model, _ = FastLanguageModel.from_pretrained(
    model_name = base_model_name, # MODEL USED FOR TRAINING
    max_seq_length = 512,
    load_in_4bit = True,
)

adapter_path = os.path.join(output_dir, "final_checkpoint")
tokenizer = AutoTokenizer.from_pretrained(adapter_path, trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, adapter_path)


#merged_model = model.merge_and_unload()

messages = [
{"role" : "user", "content" : """
Solve the issue:
my_arr=[1,2,3,4,5,6,7,8,9]
i=0
while i>len(my_arr):
    print(my_arr[i])
    i=i-1"""}
]

text = tokenizer.apply_chat_template(
messages,
tokenize = False,
add_generation_prompt = True,
enable_thinking = False,
)

from transformers import TextStreamer
_ = model.generate(
**tokenizer(text, return_tensors = "pt").to("cuda"),
max_new_tokens = 256, # Increase for longer outputs!
temperature = 0.7, top_p = 0.8, top_k = 20, # For non thinking
streamer = TextStreamer(tokenizer, skip_prompt = True),
)

<h3>Doing inference with the model from hugging face using langchain</h3>

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import ChatHuggingFace, HuggingFacePipeline
model_id="TheCasvi/Qwen3-4B-CodeMedic-adapter"
llm = HuggingFacePipeline.from_model_id(
    model_id=model_id,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 1000,
        "do_sample": False,
        "repetition_penalty": 1.03,
    }
)
chat_model = ChatHuggingFace(llm=llm, model_id=model_id)
messages = [
    SystemMessage(content="You're a helpful code assistant"),
    HumanMessage(
        content="""Solve the issue:
my_arr=[1,2,3,4,5,6,7,8,9]
i=0
while i>len(my_arr):
    print(my_arr[i])
    i=i-1"""""
    ),
]

ai_msg = chat_model.invoke(messages)

print(ai_msg.content)